# bayesflow_hpo — Getting Started

A minimal end-to-end example that shows how to run **hyperparameter optimization** (HPO) for a [BayesFlow 2.x](https://bayesflow.org) amortized inference workflow.

We will:

1. Define a simple Gaussian **simulator** (prior + likelihood).
2. Build a BayesFlow **adapter** that maps raw simulation output to the format expected by the neural network.
3. Launch an Optuna-backed **multi-objective optimization** run that searches over network architectures and training hyperparameters.
4. Inspect results: Pareto front, hyperparameter importance, and metric summaries.
5. **Select the best configuration and retrain** with a full training budget for production use.

## 0. Setup

Install the package (editable mode from the repo root) and import the two libraries we need:
- **`bayesflow`** — the core amortized Bayesian inference framework (simulators, adapters, workflows),
- **`bayesflow_hpo`** — this package, which adds HPO search spaces, objectives, and validation utilities on top.

In [ ]:
%pip install --quiet --upgrade -e ..

import bayesflow as bf
import bayesflow_hpo as hpo
import numpy as np

## 1. Simulator & Adapter

**Simulator** — We define a toy generative model with a 1-D Gaussian prior $\theta \sim \mathcal{N}(0, 1)$ and a Gaussian likelihood $x_i \mid \theta \sim \mathcal{N}(\theta, 1)$ producing 12 observations per dataset. This is deliberately simple so the notebook runs in seconds.

**Adapter** — The `bf.Adapter` tells BayesFlow how to reshape the raw simulation dictionaries into the tensor format the neural network expects:
- `.as_set(["x"])` marks observation vectors as exchangeable (order doesn't matter),
- `.rename("theta", "inference_variables")` maps the parameter to the inference target,
- `.concatenate(["x"], into="summary_variables")` stacks observations into the summary input.

`optimize()` automatically infers `param_keys` and `data_keys` from the adapter's `Rename`/`Concatenate` transforms, and generates a fixed validation dataset internally — so we only need the simulator and adapter.

In [ ]:
def prior_fn():
    return {"theta": np.random.normal(0.0, 1.0, size=(1,)).astype("float32")}


def likelihood_fn(theta):
    theta_value = float(np.squeeze(theta))
    x = np.random.normal(theta_value, 1.0, size=(12, 1)).astype("float32")
    return {"x": x}


simulator = bf.simulators.make_simulator([prior_fn, likelihood_fn])
adapter = (
    bf.Adapter()
    .as_set(["x"])
    .rename("theta", "inference_variables")
    .concatenate(["x"], into="summary_variables", axis=-1)
)

## 2. Run HPO

`hpo.optimize` is the main entry point. Under the hood it:

1. **Infers keys** from the adapter — `param_keys=["theta"]` and `data_keys=["x"]` are detected automatically from the `Rename`/`Concatenate` transforms.
2. **Generates a fixed validation dataset** — drawn once from the simulator and reused across all trials for fair comparison.
3. **Creates an Optuna study** with three objectives (in `"pareto"` mode): *calibration_error* (minimize), *nrmse* (minimize), and *inference_time* (minimize) — a Pareto-style trade-off between accuracy metrics and computational cost.
4. **Samples hyperparameters** from a `CompositeSearchSpace` that covers the inference network (flow matching), the summary network (DeepSet), and training settings (learning rate).
5. **Builds, trains, and validates** a fresh `bf.BasicWorkflow` for each trial, using SBC-based metrics on the fixed validation dataset.
6. **Reports results** back to Optuna, which guides future sampling via its TPE (Tree-structured Parzen Estimator) sampler.

Key arguments in this example:
| Argument | Value | Why |
|---|---|---|
| `n_trials` | 5 | Number of HPO configurations to try (increase for real use) |
| `epochs` | 30 | Enough gradient steps for meaningful metric signal |
| `batches_per_epoch` | 30 | 900 gradient steps per trial — fast demo |
| `max_param_count` | 500,000 | Reject very large models before training |
| `objective_metrics` | `["calibration_error", "nrmse"]` | Combines coverage quality with point-estimate accuracy |
| `objective_mode` | `"pareto"` | Each metric becomes a separate Pareto objective (plus the cost metric) |
| `train_fn` | compatibility hook | Maps `batches_per_epoch` to `num_batches` for BayesFlow versions that require it |
| `storage` | `None` | In-memory study — no leftover database files |

After optimization, `study.best_trials` returns the Pareto-optimal trial(s).

In [ ]:
import bayesflow_hpo as hpo

# Use a focused search space for the demo (flow matching + deep set).
search_space = hpo.CompositeSearchSpace(
    inference_space=hpo.FlowMatchingSpace(),
    summary_space=hpo.DeepSetSpace(),
    training_space=hpo.TrainingSpace(),
)

# BayesFlow 2.0.8 expects `num_batches` in fit(); map from HPO's `batches_per_epoch`.
def train_fn(approximator, simulator, hparams, callbacks):
    approximator.fit(
        simulator=simulator,
        epochs=int(hparams["epochs"]),
        batch_size=int(hparams.get("batch_size", 256)),
        num_batches=int(hparams["batches_per_epoch"]),
        callbacks=callbacks,
    )

study = hpo.optimize(
    # Model
    simulator=simulator,
    adapter=adapter,
    # Search
    search_space=search_space,
    n_trials=5,
    epochs=30,
    batches_per_epoch=30,
    max_param_count=500_000,
    # Objectives
    objective_metrics=["calibration_error", "nrmse"],
    objective_mode="pareto",
    # Use compatibility train hook for current BayesFlow fit() kwargs
    train_fn=train_fn,
    # In-memory study (no leftover .db files)
    storage=None,
    show_progress_bar=False,
)

print(f"Trials: {len(study.trials)}")
print(f"Pareto-optimal trials: {len(study.best_trials)}")
if study.best_trials:
    best = min(
        study.best_trials,
        key=lambda t: t.user_attrs.get("calibration_error", float("inf")),
    )
    its = best.user_attrs.get("inference_time_s", float("nan"))
    print(
        f"Best trial by calibration_error: "
        f"{best.user_attrs.get('calibration_error', float('nan')):.4f} "
        f"(nrmse={best.user_attrs.get('nrmse', float('nan')):.4f}, "
        f"inference_time_s={its:.4f}s)"
    )

## 3. Inspect Results

### 3.1 Study Overview

`plot_study()` produces a 2x2 grid using the first two objectives: Pareto front, optimization history, parameter importance, and metric scatter.

For studies with 3+ objectives, use `plot_pareto_3d()` and `plot_parallel_coordinates()` separately for full-dimensional views.

In [ ]:
fig = hpo.plot_study(study)

### 3.2 Individual Plot Functions

For finer control, call plot functions directly. Each accepts an optional `ax` argument for embedding into custom layouts:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
hpo.plot_pareto_front(study, ax=ax)
plt.show()

### 3.5 Tabular Summary

`trials_to_dataframe` converts completed trials into a `pandas.DataFrame` with one row per trial and columns for every hyperparameter, objective values, and validation metrics (NRMSE, correlation, coverage, etc.). Budget-rejected trials are excluded by default.

The table is sorted by calibration error (the first objective) so the best architectures appear at the top.

In [ ]:
df = hpo.trials_to_dataframe(study)
print(f"Completed trials: {len(df)}")

# Show the most informative columns, sorted by calibration error
key_cols = ["trial_number", "calibration_error", "nrmse", "inference_time_s",
            "param_count", "correlation", "coverage_90", "coverage_95", "training_time_s"]
display_cols = [c for c in key_cols if c in df.columns]
df.sort_values("calibration_error")[display_cols]

### 3.6 Study Summary

`summarize_study` prints a concise overview: trial counts (trained, rejected, pruned, failed), the Pareto front size, and the best trial's objective values with time in seconds. For detailed results use `trial_table()`; for hyperparameters use `best_config()`.

In [ ]:
print(hpo.summarize_study(study))

## 4. Model Selection & Retraining

HPO gives you a ranked set of configurations — but the models trained during search used a small budget (few epochs, few batches) just to *compare* architectures. To get a production-quality model you need to:

1. **Inspect** the results and pick a trial,
2. **Extract** its hyperparameters,
3. **Rebuild** a fresh (untrained) approximator from those hyperparameters,
4. **Retrain** with a full training budget,
5. **Save** the model and metadata for reproducibility.

### 4.1 Ranked Trial Table

`trial_table()` returns a ranked DataFrame sorted by the first objective (calibration error). Use it to compare the top candidates before choosing one.

In [ ]:
table = hpo.trial_table(study, top_k=5, metrics=["correlation", "coverage_90"])
table

### 4.2 Extract Hyperparameters

Once you've picked a trial, `best_config()` extracts its hyperparameters as a dictionary. Called without a `trial_number` it auto-selects the best trial by the first objective. To pick a specific trial pass `trial_number=N`.

In [ ]:
config = hpo.best_config(study)

### 4.3 Rebuild & Retrain

`build_continuous_approximator()` constructs an approximator from the hyperparameter config. Pass `checkpoint_dir` to warm-start from a trial's saved weights (e.g. from the `CheckpointPool`), or omit it to start from scratch.

We then compile with an Adam optimizer using a cosine-decay learning rate schedule and retrain with a larger budget than HPO used (2 500 gradient steps vs. 900 during search).

In [ ]:
import keras

# Optionally warm-start from the best trial's checkpoint weights.
# checkpoint_dir = pool.best_checkpoint_dir  # if a CheckpointPool was used
approximator = hpo.build_continuous_approximator(config, adapter, search_space)

# Compile with cosine-decay learning rate
epochs, batches_per_epoch = 50, 50
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=float(config["initial_lr"]),  # best_config() returns str
    decay_steps=epochs * batches_per_epoch,
)
approximator.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule))

# Retrain with a larger budget than HPO used (num_batches for BayesFlow 2.0.8+)
approximator.fit(
    simulator=simulator,
    epochs=epochs,
    num_batches=batches_per_epoch,
    batch_size=int(config.get("batch_size", 256)),
)

### 4.4 Save the Final Model

`save_workflow_with_metadata()` persists the trained approximator as a `.keras` file alongside a `.json` metadata sidecar that records the hyperparameters, library versions, and any extra info you provide. This makes the model fully reproducible.

In [ ]:
metadata = hpo.get_workflow_metadata(
    config=config,
    model_type="ContinuousApproximator",
    extra={"retrained_epochs": epochs, "retrained_batches_per_epoch": batches_per_epoch},
)
model_path = hpo.save_workflow_with_metadata(approximator, "quickstart_model", metadata)
print(f"Model saved to {model_path}")
print(f"Metadata saved to {model_path.with_suffix('.json')}")

## Done!

This notebook walked through the full HPO-to-production workflow:

1. **Defined** a simulator and adapter,
2. **Ran** multi-objective HPO over network architectures and training hyperparameters,
3. **Inspected** results (Pareto front, importance, metrics),
4. **Selected** the best configuration, **retrained** with a full budget, and **saved** the model.

For real projects you would also **validate the retrained model** (e.g. run SBC on held-out data) and compare its metrics against the HPO validation scores to confirm that the longer training improved or maintained quality. See `bayesflow_hpo.run_validation_pipeline()` for details.